# Стоковая YOLOv8n → RKNN INT8 для Luckfox Pico Ultra RV1106

Вариант исходного блокнота без обучения и без загрузки размеченного датасета.
Используются готовые веса **yolov8n.pt (COCO)**. На выходе — **yolov8n_rv1106_int8.rknn** и **labels.txt**.

Выполняйте ячейки сверху вниз в **новом сеансе Google Colab**. GPU не обязателен.
Нужен интернет для загрузки весов, зависимостей и репозиториев.
Версия RKNN-Toolkit2 оставлена из исходного блокнота (2.3.2); совместимость с runtime вашей прошивки нужно проверить на плате.

## Люди и машины

У стоковой модели 80 классов. Нужные исходные ID: **0 = person**, **2 = car** (легковой автомобиль).
В этом блокноте выходные классы сети не обрезаются: так сохраняется формат штатного C++-постпроцессора из туториала.
**Полученный RKNN сам по себе всё ещё распознаёт 80 классов.** На плате необходимо пропускать дальше только detections с ID 0 и 2 — перед трекингом, подсчётом, отрисовкой и тревогами. Инструкция есть в конце.

Не сокращайте labels.txt до двух строк и не меняйте OBJ_CLASS_NUM на 2:
это не фильтр и нарушит соответствие выходам стоковой сети.
Изменять архитектуру или обучать новую голову детектора здесь не требуется.

## Зачем остаются изображения

INT8-квантизации нужны изображения для калибровки диапазонов чисел.
Это **не обучение**, разметка не используется. Для пробной конвертации берутся 20 готовых изображений из RKNN Model Zoo.
Для проверки качества в реальных условиях лучше повторить калибровку на дневных и ночных кадрах вашей камеры
(путь CALIBRATION_DIR в ячейке конвертации). Обучать модель при этом не требуется.


## 1. Зависимости и параметры


In [ ]:
# Устанавливаем зависимости исходного блокнота.
# Проверка GPU, параметры DEVICE/BATCH и настройки обучения удалены:
# загрузка стоковых весов и экспорт выполняются без обучения.
%pip install -q ultralytics pyyaml

from pathlib import Path
import os
import sys
import subprocess

IMG_SIZE = 640
TARGET_PLATFORM = "rv1106"  # Pico Ultra; в исходном блокноте был rv1103.
MODEL_PT = "/content/yolov8n.pt"
RKNN_OUT = "/content/yolov8n_rv1106_int8.rknn"


## 2. Загрузка готовой YOLOv8n


In [ ]:
from ultralytics import YOLO

%cd /content

# YOLO скачает официальные предобученные веса, если файла ещё нет.
# Здесь нет вызова model.train(): используются готовые знания модели COCO.
# Файл с таким именем в /content должен быть именно официальными весами.
model = YOLO(MODEL_PT)
assert Path(MODEL_PT).is_file(), "Не удалось получить yolov8n.pt"

# Сохраняем ВСЕ названия в исходном порядке, чтобы labels.txt совпадал
# с 80 выходными классами сети и OBJ_CLASS_NUM в C++.
CLASS_NAMES = [model.names[i] for i in range(len(model.names))]
assert len(CLASS_NAMES) == 80, "Ожидалась стоковая COCO-модель с 80 классами"
assert CLASS_NAMES[0] == "person" and CLASS_NAMES[2] == "car"
print("Веса:", MODEL_PT)
print("Для приложения на плате: 0 = person; 2 = car.")
print("Сам RKNN сохранит все", len(CLASS_NAMES), "классов.")


## 3. Экспорт ONNX через форк Rockchip

Сохранён способ экспорта исходного блокнота. Патч выбирает прежний ONNX-экспортёр на новых версиях PyTorch.


In [ ]:
from pathlib import Path
import re
import os
import sys
import subprocess

%cd /content
# Используем тот же форк Rockchip, что и в исходном блокноте.
# При повторном запуске не удаляем уже скачанные файлы.
if not Path("/content/ultralytics_yolov8").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/airockchip/ultralytics_yolov8",
         "/content/ultralytics_yolov8"], check=True
    )
!pip install -q "onnx==1.16.1" "onnxruntime==1.19.2" "onnxslim"

%cd /content/ultralytics_yolov8

# 1) Меняем только параметры экспорта: стоковые веса, размер входа,
# формат Rockchip и batch=1. Обучение и data.yaml здесь не нужны.
cfg = Path("ultralytics/cfg/default.yaml")
text = cfg.read_text(encoding="utf-8")
text = re.sub(r"(?m)^model:.*$", f"model: {MODEL_PT}", text)
text = re.sub(r"(?m)^imgsz:.*$", f"imgsz: {IMG_SIZE}", text)
text = re.sub(r"(?m)^format:.*$", "format: rknn", text)
text = re.sub(r"(?m)^batch:.*$", "batch: 1", text)
# Явно экспортируем на CPU, даже если в Colab выбран GPU.
text = re.sub(r"(?m)^device:.*$", "device: cpu", text)
cfg.write_text(text, encoding="utf-8")
print("default.yaml:")
!grep -E "^(model|imgsz|format|batch):" ultralytics/cfg/default.yaml

# 2) Сохраняем патч исходного блокнота для нового PyTorch:
# dynamo=False выбирает прежний путь ONNX-экспорта.
# Изменение применяется только к export_rknn(), а не к весам модели.
exp = Path("ultralytics/engine/exporter.py")
src = exp.read_text(encoding="utf-8")
needle = "input_names=['images'])"
# только первый вызов в export_rknn
idx = src.find("def export_rknn")
assert idx >= 0, "export_rknn не найден"
replacement = "input_names=['images'],\n            dynamo=False)"
if replacement not in src[idx:]:
    idx2 = src.find(needle, idx)
    assert idx2 >= 0, "Форк изменился: нужно проверить export_rknn"
    src = src[:idx2] + replacement + src[idx2 + len(needle):]
    exp.write_text(src, encoding="utf-8")
print("export_rknn: dynamo=False установлен")

# Запускаем экспорт отдельным процессом, чтобы использовался форк,
# а не уже импортированный выше пакет ultralytics.
export_env = os.environ.copy()
export_env["PYTHONPATH"] = "/content/ultralytics_yolov8"
# Старый форк загружает полный PyTorch checkpoint. На PyTorch >= 2.6
# эта переменная восстанавливает прежний способ загрузки для процесса
# экспорта. Здесь она предназначена только для официальных yolov8n.pt.
export_env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
subprocess.run(
    [sys.executable, "./ultralytics/engine/exporter.py"],
    env=export_env, check=True
)

# Rockchip format=rknn на этом шаге создаёт ONNX, а не готовый RKNN.
ONNX_PATH = str(Path(MODEL_PT).with_suffix(".onnx"))
assert Path(ONNX_PATH).is_file(), f"ONNX не создан: {ONNX_PATH}"
import onnx
onnx.checker.check_model(onnx.load(ONNX_PATH))
!ls -lh "{ONNX_PATH}"
print("ONNX_PATH =", ONNX_PATH)


## 4. Конвертация ONNX → RKNN INT8 для RV1106


In [ ]:
from pathlib import Path
import re
import subprocess
import sys

%cd /content
# Версия сохранена из исходного блокнота. Если pip сообщает о конфликте
# или отсутствии wheel для Python в Colab, остановитесь и изучите ошибку.
%pip install -q "rknn-toolkit2==2.3.2"

if not Path("/content/rknn_model_zoo").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/airockchip/rknn_model_zoo",
         "/content/rknn_model_zoo"], check=True
    )

# Новый источник калибровки вместо train/images и DATASET_YAML.
# Пустая строка = готовые 20 изображений Rockchip для пробной конвертации.
# Чтобы использовать свои кадры БЕЗ разметки, загрузите их через панель
# файлов Colab в отдельную папку и впишите путь, например
# CALIBRATION_DIR = "/content/calibration".
CALIBRATION_DIR = ""

if CALIBRATION_DIR:
    image_dir = Path(CALIBRATION_DIR)
    assert image_dir.is_dir(), f"Нет папки: {image_dir}"
    calib = sorted(
        p.resolve() for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
else:
    coco_dir = Path("/content/rknn_model_zoo/datasets/COCO")
    source_list = coco_dir / "coco_subset_20.txt"
    calib = [
        (coco_dir / line.strip()).resolve()
        for line in source_list.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

assert calib, "Нет изображений для INT8-калибровки"
assert all(p.is_file() for p in calib), "Некоторые изображения отсутствуют"

# Проверяем, что скачаны изображения, а не текстовые указатели Git LFS.
from PIL import Image
for path in calib:
    with Image.open(path) as image:
        image.verify()

# Абсолютные пути исключают зависимость от рабочей папки convert.py.
subset = Path("/content/data_subset.txt")
subset.write_text("\n".join(str(p) for p in calib) + "\n", encoding="utf-8")
print("Калибровка (не обучение):", len(calib), "изображений")

# Сохраняем используемый в исходнике конвертер Rockchip.
conv = Path("/content/rknn_model_zoo/examples/yolov8/python/convert.py")
src = conv.read_text(encoding="utf-8")
src, replacements = re.subn(
    r"(?m)^DATASET_PATH\s*=\s*[^\n]+",
    "DATASET_PATH = '/content/data_subset.txt'",
    src, count=1,
)
assert replacements == 1, "Конвертер изменился: DATASET_PATH не найден"
conv.write_text(src, encoding="utf-8")

# i8 включает INT8-квантизацию; целевая платформа теперь RV1106.
# check=True останавливает ячейку при ошибке, чтобы старый файл не
# был ошибочно принят за результат успешной новой конвертации.
subprocess.run(
    [sys.executable, str(conv), ONNX_PATH, TARGET_PLATFORM, "i8", RKNN_OUT],
    cwd=str(conv.parent), check=True
)
assert Path(RKNN_OUT).is_file() and Path(RKNN_OUT).stat().st_size > 0
print("RKNN создан:", RKNN_OUT)
print("Это подтверждает конвертацию; качество и FPS проверяются на плате.")


## Скачать модель и названия классов


In [ ]:
from google.colab import files
from pathlib import Path

# Оставляем исходный способ формирования labels.txt.
# У стоковой модели здесь 80 строк: person — строка 1, car — строка 3.
# ID считаются с нуля; менять порядок или удалять другие строки нельзя.
labels_path = Path("/content/labels.txt")
labels_path.write_text("\n".join(CLASS_NAMES) + "\n", encoding="utf-8")
assert len(CLASS_NAMES) == 80
assert Path(RKNN_OUT).is_file(), "Сначала завершите конвертацию"

files.download(RKNN_OUT)
files.download(str(labels_path))
print("Скачайте оба файла. ONNX и PT остаются в /content для отладки.")
print("На плате ещё нужно включить фильтрацию ID 0 и 2; см. следующую ячейку.")


## 5. Как оставить в приложении только людей и легковые автомобили

В этом блокноте нет исходников программы Luckfox, поэтому её фильтр здесь не изменён.
Скачанный RKNN содержит **80 классов**, а не два.

При сборке программы по туториалу:

1. Передайте полученные `yolov8n_rv1106_int8.rknn` и полный `labels.txt`.
2. Оставьте `OBJ_CLASS_NUM = 80`, если это поле присутствует в используемом постпроцессоре.
3. В цикле обработки готовых детекций пропускайте любую детекцию, чей class ID не равен **0** или **2**.
   Применяйте этот фильтр до подсчёта объектов, OLED/LED, отрисовки рамок, передачи в трекер и формирования событий.
4. Не путайте class ID (0 = человек, 2 = автомобиль) с индивидуальным track ID человека.
5. Уточните пути модели и labels.txt в вашей сборке. Размер входа — 640×640; выходы этого ONNX предназначены для Rockchip-постпроцессора.
6. Проверьте runtime и NPU-драйвер вашей прошивки вместе с этой RKNN-моделью.

`classes=[0, 2]` — параметр фильтрации при predict/track в Ultralytics на ПК.
Он не встраивается автоматически в RKNN, не меняет 80 выходных классов и не настраивает программу HelloDetector.

Класс car не включает все виды транспорта: truck имеет ID 7, bus — 5, motorcycle — 3.
В этой задаче под «машинами» принят класс car.

После такого фильтра приложение использует только людей и автомобили, хотя сеть вычисляет оценки всех COCO-классов.
Для RKNN с физически двумя выходными классами потребовалось бы отдельное изменение графа и постпроцессора;
здесь оно намеренно не выполняется, чтобы сохранить стоковую модель и существующий путь экспорта.

## Что изменено относительно исходного блокнота

Удалены продолжение обучения из best.pt, проверка GPU для обучения, загрузка my_dataset.zip,
переписывание data.yaml, model.train() и model.val() на пользовательском датасете.
Сохранены форк Rockchip, настройка default.yaml, патч dynamo=False, конвертер convert.py и скачивание результатов.
Необходимые изменения: загрузка готовых весов, правильный путь ONNX, калибровочные изображения вместо train,
rv1106 вместо rv1103 и проверки успешности процессов. Изменённые места снабжены комментариями.

Этот блокнот подготовлен и статически проверен; сквозная конвертация в Colab и запуск на Luckfox
ещё не выполнены. Репозитории клонируются с текущей ветки, как в исходном файле; при их изменении
может понадобиться обновить зависимости или патч экспортёра.

Источники:
- [Исходный туториал](https://github.com/Daniilslep/LuckFox_YOLO)
- [Форк экспорта YOLOv8 Rockchip](https://github.com/airockchip/ultralytics_yolov8)
- [RKNN Model Zoo](https://github.com/airockchip/rknn_model_zoo)
